***Full subject features data***

With MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_acoustic_full = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset.csv')
print(df_acoustic_full.shape)
print(df_acoustic_full.head())

(193, 71)
  Participant_ID  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  \
0           d1_1   118.719333    141.312833    106.029833        0.010085   
1          d1_10   113.014000    154.201333    105.652667        0.008043   
2          d1_11   223.657833    239.034500    161.921500        0.002585   
3          d1_12   141.336667    165.070500     66.233833        0.009008   
4          d1_13   112.839667    208.860000    107.392833        0.004850   

   MDVP:Jitter(Abs)  MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  ...  \
0          0.000087  0.004998  0.007312    0.014992      0.053932  ...   
1          0.000072  0.003508  0.005035    0.010527      0.029677  ...   
2          0.000010  0.001463  0.001425    0.004393      0.017093  ...   
3          0.000065  0.005090  0.004368    0.015263      0.042640  ...   
4          0.000042  0.002468  0.002817    0.007405      0.025015  ...   

   Delta4  Delta5  Delta6  Delta7  Delta8  Delta9  Delta10  Delta11  Delta12  \
0 

Modality Acoustic-MIRA-Net

In [3]:
# ============================================================
#     SIMPLE + STABLE DANN + SCL + MLP CLASSIFIER
#     CROSS-MODAL + CROSS-PARTICIPANT ANALYSIS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Function
import torch.nn.functional as F

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# LOAD DATA
# ============================================================

# First 7 columns -> Features
# Last column     -> PD / HC labels

# ============================================================
# SELECT FEATURES AND LABEL
# ============================================================

X_acoustic = (
    df_acoustic_full
    .drop(columns=["Participant_ID", "Status"])
    .values
    .astype(np.float32)
)

y_acoustic = (
    df_acoustic_full["Status"]
    .values
    .astype(np.int64)
)

# ============================================================
# STANDARDIZATION
# ============================================================

scaler_acoustic = StandardScaler()
scaler_meg = StandardScaler()

X_acoustic = scaler_acoustic.fit_transform(X_acoustic)


# ============================================================
# DOMAIN LABELS
# ============================================================

# 0 -> Acoustic
# 1 -> MEG

domain_acoustic = np.zeros(len(X_acoustic))


# ============================================================
# COMBINE
# ============================================================

X_all = np.vstack([
    X_acoustic

]).astype(np.float32)

task_labels = np.hstack([
    y_acoustic

]).astype(np.int64)

domain_labels = np.hstack([
    domain_acoustic

]).astype(np.int64)

# ============================================================
# TENSORS
# ============================================================

X_all_tensor = torch.tensor(
    X_all
).float().to(device)

task_labels_tensor = torch.tensor(
    task_labels
).long().to(device)

domain_labels_tensor = torch.tensor(
    domain_labels
).long().to(device)

# ============================================================
# GRADIENT REVERSAL
# ============================================================

class GradReverse(Function):

    @staticmethod
    def forward(ctx, x, lambd):

        ctx.lambd = lambd

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):

        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):

    return GradReverse.apply(x, lambd)

# ============================================================
# SIMPLE RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(dim, dim),
            nn.ReLU(),

            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )

# ============================================================
# DANN + SCL MODEL
# ============================================================

class DANNContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        # ====================================================
        # SIMPLE ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        # ====================================================
        # TASK HEAD
        # ====================================================

        self.task_head = nn.Sequential(

            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        # ====================================================
        # DOMAIN HEAD
        # ====================================================

        self.domain_head = nn.Sequential(

            nn.Linear(latent_dim, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        # L2 NORMALIZATION

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        # DANN

        rev_z = grad_reverse(
            z_norm,
            lambd
        )

        task_out = self.task_head(z_norm)

        domain_out = self.domain_head(rev_z)

        return task_out, domain_out, z_norm

# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()

        self.temperature = temperature

    def forward(self, features, labels):

        device = features.device

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(device)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity,
            dim=1,
            keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = torch.ones_like(mask) - torch.eye(
            mask.shape[0]
        ).to(device)

        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(1, keepdim=True) + 1e-8
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(1) / (mask.sum(1) + 1e-8)

        loss = -mean_log_prob_pos.mean()

        return loss

# ============================================================
# DATA LOADER
# ============================================================

dataset = TensorDataset(
    X_all_tensor,
    task_labels_tensor,
    domain_labels_tensor
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# ============================================================
# MODEL
# ============================================================

model = DANNContrastive(
    input_dim=X_all.shape[1],
    latent_dim=32
).to(device)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# LOSSES
# ============================================================

task_loss_fn = nn.CrossEntropyLoss()

domain_loss_fn = nn.CrossEntropyLoss()

contrastive_loss_fn = SupConLoss(
    temperature=0.07
)

epochs = 120

print("\nTraining Started...\n")

# ============================================================
# TRAINING
# ============================================================

for epoch in range(epochs):

    model.train()

    total_correct = 0
    total_samples = 0

    # ========================================================
    # WEAKER DANN
    # ========================================================

    p = min(epoch / 50.0, 1.0)

    lambd = 0.25 * (
        2 / (1 + np.exp(-10 * p)) - 1
    )

    for xb, y_task, y_dom in loader:

        optimizer.zero_grad()

        task_out, dom_out, z = model(
            xb,
            lambd
        )

        # ====================================================
        # LOSSES
        # ====================================================

        loss_task = task_loss_fn(
            task_out,
            y_task
        )

        loss_domain = domain_loss_fn(
            dom_out,
            y_dom
        )

        loss_contrastive = contrastive_loss_fn(
            z,
            y_task
        )

        # ====================================================
        # FINAL LOSS
        # ====================================================

        loss = (
            3.0 * loss_task +
            0.25 * loss_domain +
            0.55 * loss_contrastive
        )

        loss.backward()

        optimizer.step()

        preds = task_out.argmax(1)

        total_correct += (
            preds == y_task
        ).sum().item()

        total_samples += len(y_task)

    acc = total_correct / total_samples

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| λ={lambd:.3f} "
        f"| Acc={acc:.4f}"
    )

# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

model.eval()

with torch.no_grad():

    fused_acoustic_32D = model(
        torch.tensor(X_acoustic).float().to(device),
        0
    )[2].cpu().numpy()




# ============================================================
# COMBINE LATENTS
# ============================================================

X_fused = np.vstack([
    fused_acoustic_32D

])

y_fused = np.hstack([
    y_acoustic

])

# ============================================================
# SIMPLE MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# CROSS PARTICIPANT EVALUATION
# ============================================================

print("\n======================================")
print(" CROSS PARTICIPANT ANALYSIS")
print("======================================\n")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

accs = []
precs = []
recs = []
f1s = []
aucs = []

fold = 1

for train_idx, test_idx in skf.split(X_fused, y_fused):

    X_train = torch.tensor(
        X_fused[train_idx]
    ).float().to(device)

    X_test = torch.tensor(
        X_fused[test_idx]
    ).float().to(device)

    y_train = torch.tensor(
        y_fused[train_idx]
    ).long().to(device)

    y_test = y_fused[test_idx]

    clf = FinalMLP().to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    # ========================================================
    # TRAIN MLP
    # ========================================================

    clf.train()

    for epoch in range(80):

        optimizer_clf.zero_grad()

        outputs = clf(X_train)

        loss = criterion(
            outputs,
            y_train
        )

        loss.backward()

        optimizer_clf.step()

    # ========================================================
    # TEST
    # ========================================================

    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test)

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        preds = outputs.argmax(1).cpu().numpy()

    acc = accuracy_score(y_test, preds)

    prec = precision_score(y_test, preds)

    rec = recall_score(y_test, preds)

    f1 = f1_score(y_test, preds)

    auc = roc_auc_score(y_test, probs)

    accs.append(acc)
    precs.append(prec)
    recs.append(rec)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold}")

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")

    print()

    fold += 1

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n======================================")
print(" FINAL RESULTS")
print("======================================\n")

print(f"Mean Accuracy  : {np.mean(accs)*100:.2f}%")
print(f"Mean Precision : {np.mean(precs)*100:.2f}%")
print(f"Mean Recall    : {np.mean(recs)*100:.2f}%")
print(f"Mean F1-Score  : {np.mean(f1s)*100:.2f}%")
print(f"Mean AUC       : {np.mean(aucs)*100:.2f}%")

print("\n======================================")
print(" LATENT SHAPES")
print("======================================\n")

print("Acoustic :", fused_acoustic_32D.shape)



Training Started...

Epoch 1/120 | λ=0.000 | Acc=0.4663
Epoch 2/120 | λ=0.025 | Acc=0.6321
Epoch 3/120 | λ=0.049 | Acc=0.6580
Epoch 4/120 | λ=0.073 | Acc=0.6580
Epoch 5/120 | λ=0.095 | Acc=0.6580
Epoch 6/120 | λ=0.116 | Acc=0.6580
Epoch 7/120 | λ=0.134 | Acc=0.6580
Epoch 8/120 | λ=0.151 | Acc=0.6580
Epoch 9/120 | λ=0.166 | Acc=0.6580
Epoch 10/120 | λ=0.179 | Acc=0.6580
Epoch 11/120 | λ=0.190 | Acc=0.6580
Epoch 12/120 | λ=0.200 | Acc=0.6580
Epoch 13/120 | λ=0.208 | Acc=0.6580
Epoch 14/120 | λ=0.215 | Acc=0.6580
Epoch 15/120 | λ=0.221 | Acc=0.6580
Epoch 16/120 | λ=0.226 | Acc=0.6580
Epoch 17/120 | λ=0.230 | Acc=0.6580
Epoch 18/120 | λ=0.234 | Acc=0.6580
Epoch 19/120 | λ=0.237 | Acc=0.6580
Epoch 20/120 | λ=0.239 | Acc=0.6580
Epoch 21/120 | λ=0.241 | Acc=0.6580
Epoch 22/120 | λ=0.243 | Acc=0.6580
Epoch 23/120 | λ=0.244 | Acc=0.6580
Epoch 24/120 | λ=0.245 | Acc=0.6580
Epoch 25/120 | λ=0.246 | Acc=0.6580
Epoch 26/120 | λ=0.247 | Acc=0.6580
Epoch 27/120 | λ=0.247 | Acc=0.6580
Epoch 28/120 | 

In [4]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)
fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()


# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_acoustic_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([193, 32])
torch.float32


In [5]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)
y_acoustic = df_acoustic_full.iloc[:, -1].values.astype(np.int32)   # (7350,)



# Convert to tensors (reshape to column)
acoustic_target_tensor = torch.tensor(y_acoustic).long().reshape(-1, 1)   # (7350, 1)
      # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()  # (7350, 32)


# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target
acoustic_full = torch.cat([fused_acoustic_tensor, acoustic_target_tensor.float()], dim=1)  # (7350, 65)


print(acoustic_full)


tensor([[-0.1052, -0.0390, -0.0399,  ...,  0.1922,  0.1769,  1.0000],
        [-0.1169, -0.0383, -0.0444,  ...,  0.1891,  0.1754,  1.0000],
        [-0.1542,  0.1968, -0.2244,  ...,  0.1827, -0.0596,  0.0000],
        ...,
        [-0.1295, -0.0213, -0.0589,  ...,  0.2099,  0.1903,  1.0000],
        [-0.1305, -0.0205, -0.0603,  ...,  0.2094,  0.1872,  1.0000],
        [-0.1272, -0.0228, -0.0575,  ...,  0.2074,  0.1922,  1.0000]])


In [6]:
acoustic_full

tensor([[-0.1052, -0.0390, -0.0399,  ...,  0.1922,  0.1769,  1.0000],
        [-0.1169, -0.0383, -0.0444,  ...,  0.1891,  0.1754,  1.0000],
        [-0.1542,  0.1968, -0.2244,  ...,  0.1827, -0.0596,  0.0000],
        ...,
        [-0.1295, -0.0213, -0.0589,  ...,  0.2099,  0.1903,  1.0000],
        [-0.1305, -0.0205, -0.0603,  ...,  0.2094,  0.1872,  1.0000],
        [-0.1272, -0.0228, -0.0575,  ...,  0.2074,  0.1922,  1.0000]])

In [7]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = acoustic_full

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(193, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0 -0.105171 -0.038995 -0.039918  0.261266 -0.286041  0.249034  0.276905   
1 -0.116949 -0.038288 -0.044410  0.257140 -0.278100  0.246120  0.270906   
2 -0.154155  0.196843 -0.224402  0.129498 -0.199062  0.175275  0.263465   
3 -0.115394 -0.030578 -0.046564  0.255736 -0.288111  0.244876  0.275136   
4 -0.110714 -0.041732 -0.048822  0.247952 -0.271808  0.256217  0.284136   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0  0.133938 -0.195474 -0.203180  ... -0.266611  0.017787 -0.303408  0.064106   
1  0.157214 -0.191127 -0.210580  ... -0.279412  0.019228 -0.302257  0.049981   
2 -0.116394 -0.257151 -0.019254  ... -0.059530  0.252291 -0.323413 -0.163807   
3  0.149855 -0.188775 -0.203595  ... -0.270562  0.024459 -0.298823  0.063518   
4  0.157095 -0.187537 -0.214540  ... -0.272039  0.031239 -0.309976  0.033771   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [8]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [9]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd

fused_df_named_int.to_csv('/content/drive//MyDrive/Modality-subj-level-full-acoustic_MIRA-Net_data.csv', index=False)

In [11]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import pandas as pd

fused_df_named_int = pd.read_csv('/content/drive//MyDrive/Modality-subj-level-full-acoustic_MIRA-Net_data.csv')

In [13]:
fused_df_named_int.head()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_23,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,PD_Label_Combined
0,-0.105171,-0.038995,-0.039918,0.261266,-0.286041,0.249034,0.276905,0.133938,-0.195474,-0.203180,...,-0.266611,0.017787,-0.303408,0.064106,-0.044580,0.215494,0.050045,0.192243,0.176866,1
1,-0.116949,-0.038288,-0.044410,0.257140,-0.278100,0.246120,0.270906,0.157214,-0.191127,-0.210580,...,-0.279412,0.019228,-0.302257,0.049981,-0.043616,0.213309,0.058503,0.189122,0.175351,1
2,-0.154155,0.196843,-0.224402,0.129498,-0.199062,0.175275,0.263465,-0.116394,-0.257151,-0.019254,...,-0.059530,0.252291,-0.323413,-0.163807,-0.221010,-0.030548,-0.145513,0.182652,-0.059644,0
3,-0.115394,-0.030578,-0.046564,0.255736,-0.288111,0.244876,0.275136,0.149855,-0.188775,-0.203595,...,-0.270562,0.024459,-0.298823,0.063518,-0.051285,0.212046,0.060703,0.186697,0.173180,1
4,-0.110714,-0.041732,-0.048822,0.247952,-0.271808,0.256217,0.284136,0.157095,-0.187537,-0.214540,...,-0.272039,0.031239,-0.309976,0.033771,-0.045757,0.206169,0.056977,0.198350,0.164807,1


In [14]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int[
    [f'feat_{i}' for i in range(32)]
].copy()

y = fused_df_named_int[
    "PD_Label_Combined"
].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00

Class Weights:
{0: 1.4621212121212122, 1: 0.7598425196850394}

Train Shape : (154, 32)
Test Shape  : (39, 32)

Training Class Distribution:
PD_Label_Combined
1    101
0     53
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    26
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(1.4528301886792452), np.int64(1): np.float64(0.7623762376237624)}

XGBoost scale_pos_weight: 0.5247524752475248


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 1.0000
Precision           : 1.0000
Sensitivity         : 1.0000
F1 Score            : 1.0000
ROC-AUC             : 1.0000
Balanced Accuracy   : 1.0000

XGBoost | Fold 2/10
Accuracy            : 1.0000
Precision           : 1.0000
Sensitivity         : 1.0000
F1 Score            : 1.0000
ROC-AUC         

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
Stacking Fusion,0.97375,0.990909,0.97,0.979449,0.996,0.975000,0.974359,0.868190,0.995459,1.000000,...,0.771905,1.00000,0.980392,0.991124,0.980769,0.944911,13.0,0.0,1.0,25.0
XGBoost,0.98000,0.990909,0.98,0.984712,0.998,0.980000,0.948718,0.831144,0.985822,0.961538,...,0.666860,0.98629,0.961538,0.992604,0.942308,0.884615,12.0,1.0,1.0,25.0
Random Forest,0.98000,0.990909,0.98,0.984712,0.998,0.980000,0.948718,0.831144,0.985822,0.961538,...,0.666860,0.98629,0.961538,0.983728,0.942308,0.884615,12.0,1.0,1.0,25.0
HistGradientBoosting,0.97375,0.981818,0.98,0.979950,0.985,0.971667,0.948718,0.831144,0.985822,0.961538,...,0.666860,0.98629,0.961538,0.958580,0.942308,0.884615,12.0,1.0,1.0,25.0
Voting Fusion,0.98000,0.990909,0.98,0.984712,0.998,0.980000,0.948718,0.831144,0.985822,0.961538,...,0.666860,0.98629,0.961538,0.985207,0.942308,0.884615,12.0,1.0,1.0,25.0


Without MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_acoustic_top7 = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset.csv')
print(df_acoustic_top7.shape)
print(df_acoustic_top7.head())

(193, 71)
  Participant_ID  MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  \
0           d1_1   118.719333    141.312833    106.029833        0.010085   
1          d1_10   113.014000    154.201333    105.652667        0.008043   
2          d1_11   223.657833    239.034500    161.921500        0.002585   
3          d1_12   141.336667    165.070500     66.233833        0.009008   
4          d1_13   112.839667    208.860000    107.392833        0.004850   

   MDVP:Jitter(Abs)  MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  ...  \
0          0.000087  0.004998  0.007312    0.014992      0.053932  ...   
1          0.000072  0.003508  0.005035    0.010527      0.029677  ...   
2          0.000010  0.001463  0.001425    0.004393      0.017093  ...   
3          0.000065  0.005090  0.004368    0.015263      0.042640  ...   
4          0.000042  0.002468  0.002817    0.007405      0.025015  ...   

   Delta4  Delta5  Delta6  Delta7  Delta8  Delta9  Delta10  Delta11  Delta12  \
0 

In [3]:
fused_df_named_int = df_acoustic_top7.rename(columns={"Status": "PD_Label_Combined"})
fused_df_named_int = fused_df_named_int.iloc[:, 1:].copy()
fused_df_named_int.head()
fused_df_named_int.shape

(193, 70)

In [4]:
fused_df_named_int.head()

,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),...,Delta4,Delta5,Delta6,Delta7,Delta8,Delta9,Delta10,Delta11,Delta12,PD_Label_Combined
0,118.719333,141.312833,106.029833,0.010085,0.000087,0.004998,0.007312,0.014992,0.053932,0.515167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,113.014000,154.201333,105.652667,0.008043,0.000072,0.003508,0.005035,0.010527,0.029677,0.288667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,223.657833,239.034500,161.921500,0.002585,0.000010,0.001463,0.001425,0.004393,0.017093,0.151333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,141.336667,165.070500,66.233833,0.009008,0.000065,0.005090,0.004368,0.015263,0.042640,0.407333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,112.839667,208.860000,107.392833,0.004850,0.000042,0.002468,0.002817,0.007405,0.025015,0.224500,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [5]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int.iloc[:, :-1].copy()

y = fused_df_named_int["PD_Label_Combined"].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00

Class Weights:
{0: 1.4621212121212122, 1: 0.7598425196850394}

Train Shape : (154, 69)
Test Shape  : (39, 69)

Training Class Distribution:
PD_Label_Combined
1    101
0     53
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    26
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(1.4528301886792452), np.int64(1): np.float64(0.7623762376237624)}

XGBoost scale_pos_weight: 0.5247524752475248


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 0.9375
Precision           : 1.0000
Sensitivity         : 0.9000
F1 Score            : 0.9474
ROC-AUC             : 0.9833
Balanced Accuracy   : 0.9500

XGBoost | Fold 2/10
Accuracy            : 0.8750
Precision           : 0.9000
Sensitivity         : 0.9000
F1 Score            : 0.9000
ROC-AUC         

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
Voting Fusion,0.791667,0.862475,0.830909,0.838327,0.907182,0.772121,0.846154,0.702705,0.927525,0.916667,...,0.577654,0.956742,0.880000,0.911243,0.846154,0.670820,11.0,2.0,4.0,22.0
HistGradientBoosting,0.747083,0.820797,0.790909,0.798699,0.885364,0.725455,0.846154,0.702705,0.927525,0.954545,...,0.666860,0.986290,0.875000,0.918639,0.865385,0.694709,12.0,1.0,5.0,21.0
Random Forest,0.791250,0.833561,0.860909,0.843752,0.895030,0.760455,0.820513,0.673320,0.910227,0.880000,...,0.497436,0.918205,0.862745,0.881657,0.807692,0.604743,10.0,3.0,4.0,22.0
XGBoost,0.816667,0.886616,0.840909,0.857194,0.900182,0.805455,0.820513,0.673320,0.910227,0.913043,...,0.577654,0.956742,0.857143,0.896450,0.826923,0.626628,11.0,2.0,5.0,21.0
Stacking Fusion,0.824167,0.911364,0.830909,0.861896,0.901667,0.820455,0.794872,0.644660,0.892203,0.909091,...,0.577654,0.956742,0.833333,0.931953,0.807692,0.585018,11.0,2.0,6.0,20.0


RF-SHAP top 7 aocutic subject data

With MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_acoustic_full = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset_top7.csv')
print(df_acoustic_full.shape)
print(df_acoustic_full.head())

(193, 9)
       RPDE  Degree-voice breaks  Delta2  Delta3  Delta11  MFCC5   spread1  \
0  0.428488                  0.0     0.0     0.0      0.0    0.0 -4.239926   
1  0.542488                  0.0     0.0     0.0      0.0    0.0 -4.519291   
2  0.332247                  0.0     0.0     0.0      0.0    0.0 -7.226125   
3  0.498593                  0.0     0.0     0.0      0.0    0.0 -5.632290   
4  0.470775                  0.0     0.0     0.0      0.0    0.0 -5.523149   

  Participant_ID  Status  
0           d1_1       1  
1          d1_10       1  
2          d1_11       0  
3          d1_12       1  
4          d1_13       1  


In [3]:
# ============================================================
#     SIMPLE + STABLE DANN + SCL + MLP CLASSIFIER
#     CROSS-MODAL + CROSS-PARTICIPANT ANALYSIS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.autograd import Function
import torch.nn.functional as F

import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# LOAD DATA
# ============================================================

# First 7 columns -> Features
# Last column     -> PD / HC labels

# ============================================================
# SELECT FEATURES AND LABEL
# ============================================================

X_acoustic = (
    df_acoustic_full
    .drop(columns=["Participant_ID", "Status"])
    .values
    .astype(np.float32)
)

y_acoustic = (
    df_acoustic_full["Status"]
    .values
    .astype(np.int64)
)

# ============================================================
# STANDARDIZATION
# ============================================================

scaler_acoustic = StandardScaler()
scaler_meg = StandardScaler()

X_acoustic = scaler_acoustic.fit_transform(X_acoustic)


# ============================================================
# DOMAIN LABELS
# ============================================================

# 0 -> Acoustic
# 1 -> MEG

domain_acoustic = np.zeros(len(X_acoustic))


# ============================================================
# COMBINE
# ============================================================

X_all = np.vstack([
    X_acoustic

]).astype(np.float32)

task_labels = np.hstack([
    y_acoustic

]).astype(np.int64)

domain_labels = np.hstack([
    domain_acoustic

]).astype(np.int64)

# ============================================================
# TENSORS
# ============================================================

X_all_tensor = torch.tensor(
    X_all
).float().to(device)

task_labels_tensor = torch.tensor(
    task_labels
).long().to(device)

domain_labels_tensor = torch.tensor(
    domain_labels
).long().to(device)

# ============================================================
# GRADIENT REVERSAL
# ============================================================

class GradReverse(Function):

    @staticmethod
    def forward(ctx, x, lambd):

        ctx.lambd = lambd

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):

        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):

    return GradReverse.apply(x, lambd)

# ============================================================
# SIMPLE RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(

            nn.Linear(dim, dim),
            nn.ReLU(),

            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )

# ============================================================
# DANN + SCL MODEL
# ============================================================

class DANNContrastive(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        # ====================================================
        # SIMPLE ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        # ====================================================
        # TASK HEAD
        # ====================================================

        self.task_head = nn.Sequential(

            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        # ====================================================
        # DOMAIN HEAD
        # ====================================================

        self.domain_head = nn.Sequential(

            nn.Linear(latent_dim, 16),
            nn.ReLU(),

            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        # L2 NORMALIZATION

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        # DANN

        rev_z = grad_reverse(
            z_norm,
            lambd
        )

        task_out = self.task_head(z_norm)

        domain_out = self.domain_head(rev_z)

        return task_out, domain_out, z_norm

# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()

        self.temperature = temperature

    def forward(self, features, labels):

        device = features.device

        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(device)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        logits_max, _ = torch.max(
            similarity,
            dim=1,
            keepdim=True
        )

        logits = similarity - logits_max.detach()

        logits_mask = torch.ones_like(mask) - torch.eye(
            mask.shape[0]
        ).to(device)

        mask = mask * logits_mask

        exp_logits = torch.exp(logits) * logits_mask

        log_prob = logits - torch.log(
            exp_logits.sum(1, keepdim=True) + 1e-8
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(1) / (mask.sum(1) + 1e-8)

        loss = -mean_log_prob_pos.mean()

        return loss

# ============================================================
# DATA LOADER
# ============================================================

dataset = TensorDataset(
    X_all_tensor,
    task_labels_tensor,
    domain_labels_tensor
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True
)

# ============================================================
# MODEL
# ============================================================

model = DANNContrastive(
    input_dim=X_all.shape[1],
    latent_dim=32
).to(device)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# ============================================================
# LOSSES
# ============================================================

task_loss_fn = nn.CrossEntropyLoss()

domain_loss_fn = nn.CrossEntropyLoss()

contrastive_loss_fn = SupConLoss(
    temperature=0.07
)

epochs = 120

print("\nTraining Started...\n")

# ============================================================
# TRAINING
# ============================================================

for epoch in range(epochs):

    model.train()

    total_correct = 0
    total_samples = 0

    # ========================================================
    # WEAKER DANN
    # ========================================================

    p = min(epoch / 50.0, 1.0)

    lambd = 0.25 * (
        2 / (1 + np.exp(-10 * p)) - 1
    )

    for xb, y_task, y_dom in loader:

        optimizer.zero_grad()

        task_out, dom_out, z = model(
            xb,
            lambd
        )

        # ====================================================
        # LOSSES
        # ====================================================

        loss_task = task_loss_fn(
            task_out,
            y_task
        )

        loss_domain = domain_loss_fn(
            dom_out,
            y_dom
        )

        loss_contrastive = contrastive_loss_fn(
            z,
            y_task
        )

        # ====================================================
        # FINAL LOSS
        # ====================================================

        loss = (
            3.0 * loss_task +
            0.25 * loss_domain +
            0.55 * loss_contrastive
        )

        loss.backward()

        optimizer.step()

        preds = task_out.argmax(1)

        total_correct += (
            preds == y_task
        ).sum().item()

        total_samples += len(y_task)

    acc = total_correct / total_samples

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"| λ={lambd:.3f} "
        f"| Acc={acc:.4f}"
    )

# ============================================================
# EXTRACT LATENT FEATURES
# ============================================================

model.eval()

with torch.no_grad():

    fused_acoustic_32D = model(
        torch.tensor(X_acoustic).float().to(device),
        0
    )[2].cpu().numpy()




# ============================================================
# COMBINE LATENTS
# ============================================================

X_fused = np.vstack([
    fused_acoustic_32D

])

y_fused = np.hstack([
    y_acoustic

])

# ============================================================
# SIMPLE MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# CROSS PARTICIPANT EVALUATION
# ============================================================

print("\n======================================")
print(" CROSS PARTICIPANT ANALYSIS")
print("======================================\n")

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

accs = []
precs = []
recs = []
f1s = []
aucs = []

fold = 1

for train_idx, test_idx in skf.split(X_fused, y_fused):

    X_train = torch.tensor(
        X_fused[train_idx]
    ).float().to(device)

    X_test = torch.tensor(
        X_fused[test_idx]
    ).float().to(device)

    y_train = torch.tensor(
        y_fused[train_idx]
    ).long().to(device)

    y_test = y_fused[test_idx]

    clf = FinalMLP().to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=1e-3
    )

    criterion = nn.CrossEntropyLoss()

    # ========================================================
    # TRAIN MLP
    # ========================================================

    clf.train()

    for epoch in range(80):

        optimizer_clf.zero_grad()

        outputs = clf(X_train)

        loss = criterion(
            outputs,
            y_train
        )

        loss.backward()

        optimizer_clf.step()

    # ========================================================
    # TEST
    # ========================================================

    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test)

        probs = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        preds = outputs.argmax(1).cpu().numpy()

    acc = accuracy_score(y_test, preds)

    prec = precision_score(y_test, preds)

    rec = recall_score(y_test, preds)

    f1 = f1_score(y_test, preds)

    auc = roc_auc_score(y_test, probs)

    accs.append(acc)
    precs.append(prec)
    recs.append(rec)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold}")

    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")

    print()

    fold += 1

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n======================================")
print(" FINAL RESULTS")
print("======================================\n")

print(f"Mean Accuracy  : {np.mean(accs)*100:.2f}%")
print(f"Mean Precision : {np.mean(precs)*100:.2f}%")
print(f"Mean Recall    : {np.mean(recs)*100:.2f}%")
print(f"Mean F1-Score  : {np.mean(f1s)*100:.2f}%")
print(f"Mean AUC       : {np.mean(aucs)*100:.2f}%")

print("\n======================================")
print(" LATENT SHAPES")
print("======================================\n")

print("Acoustic :", fused_acoustic_32D.shape)



Training Started...

Epoch 1/120 | λ=0.000 | Acc=0.3731
Epoch 2/120 | λ=0.025 | Acc=0.4715
Epoch 3/120 | λ=0.049 | Acc=0.6062
Epoch 4/120 | λ=0.073 | Acc=0.6684
Epoch 5/120 | λ=0.095 | Acc=0.6477
Epoch 6/120 | λ=0.116 | Acc=0.6321
Epoch 7/120 | λ=0.134 | Acc=0.6425
Epoch 8/120 | λ=0.151 | Acc=0.6528
Epoch 9/120 | λ=0.166 | Acc=0.6477
Epoch 10/120 | λ=0.179 | Acc=0.6580
Epoch 11/120 | λ=0.190 | Acc=0.6632
Epoch 12/120 | λ=0.200 | Acc=0.6632
Epoch 13/120 | λ=0.208 | Acc=0.6580
Epoch 14/120 | λ=0.215 | Acc=0.6580
Epoch 15/120 | λ=0.221 | Acc=0.6580
Epoch 16/120 | λ=0.226 | Acc=0.6580
Epoch 17/120 | λ=0.230 | Acc=0.6580
Epoch 18/120 | λ=0.234 | Acc=0.6580
Epoch 19/120 | λ=0.237 | Acc=0.6580
Epoch 20/120 | λ=0.239 | Acc=0.6580
Epoch 21/120 | λ=0.241 | Acc=0.6580
Epoch 22/120 | λ=0.243 | Acc=0.6580
Epoch 23/120 | λ=0.244 | Acc=0.6684
Epoch 24/120 | λ=0.245 | Acc=0.6684
Epoch 25/120 | λ=0.246 | Acc=0.6580
Epoch 26/120 | λ=0.247 | Acc=0.6580
Epoch 27/120 | λ=0.247 | Acc=0.6580
Epoch 28/120 | 

In [4]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)
fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()


# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_acoustic_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([193, 32])
torch.float32


In [5]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)
y_acoustic = df_acoustic_full.iloc[:, -1].values.astype(np.int32)   # (7350,)



# Convert to tensors (reshape to column)
acoustic_target_tensor = torch.tensor(y_acoustic).long().reshape(-1, 1)   # (7350, 1)
      # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()  # (7350, 32)


# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target
acoustic_full = torch.cat([fused_acoustic_tensor, acoustic_target_tensor.float()], dim=1)  # (7350, 65)


print(acoustic_full)


tensor([[ 0.2651, -0.0103,  0.1151,  ...,  0.0575, -0.0215,  1.0000],
        [ 0.2635, -0.0142,  0.1152,  ...,  0.0680, -0.0333,  1.0000],
        [ 0.2354, -0.1751, -0.0474,  ..., -0.0948,  0.1573,  0.0000],
        ...,
        [ 0.2636, -0.0309,  0.1020,  ...,  0.0731, -0.0473,  1.0000],
        [ 0.2637, -0.0310,  0.1017,  ...,  0.0728, -0.0471,  1.0000],
        [ 0.2640, -0.0318,  0.0996,  ...,  0.0709, -0.0454,  1.0000]])


In [6]:
acoustic_full

tensor([[ 0.2651, -0.0103,  0.1151,  ...,  0.0575, -0.0215,  1.0000],
        [ 0.2635, -0.0142,  0.1152,  ...,  0.0680, -0.0333,  1.0000],
        [ 0.2354, -0.1751, -0.0474,  ..., -0.0948,  0.1573,  0.0000],
        ...,
        [ 0.2636, -0.0309,  0.1020,  ...,  0.0731, -0.0473,  1.0000],
        [ 0.2637, -0.0310,  0.1017,  ...,  0.0728, -0.0471,  1.0000],
        [ 0.2640, -0.0318,  0.0996,  ...,  0.0709, -0.0454,  1.0000]])

In [7]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = acoustic_full

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(193, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0  0.265117 -0.010316  0.115100  0.328323  0.248238 -0.356049 -0.257898   
1  0.263472 -0.014242  0.115215  0.324907  0.249789 -0.339662 -0.261713   
2  0.235450 -0.175083 -0.047387  0.284162  0.299548 -0.321362 -0.242467   
3  0.265189 -0.012594  0.123403  0.323801  0.251000 -0.348127 -0.249223   
4  0.265249 -0.010860  0.124201  0.326514  0.248276 -0.352222 -0.249442   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0  0.174850  0.129193  0.233881  ... -0.247454  0.119079 -0.095544 -0.064415   
1  0.168203  0.135401  0.235584  ... -0.245267  0.121652 -0.108255 -0.057349   
2 -0.029511  0.056156  0.208431  ... -0.348995 -0.075789 -0.172913  0.116194   
3  0.160804  0.132252  0.234291  ... -0.251133  0.107482 -0.113234 -0.056090   
4  0.164312  0.131111  0.233590  ... -0.253278  0.106324 -0.109191 -0.057897   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [8]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [9]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd

fused_df_named_int.to_csv('/content/drive//MyDrive/Modality-subj-level-top7(1)-acoustic_MIRA-Net_data.csv', index=False)

In [11]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
fused_df_named_int = pd.read_csv('/content/drive//MyDrive/Modality-subj-level-top7(1)-acoustic_MIRA-Net_data.csv')

In [13]:
fused_df_named_int.head()

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_23,feat_24,feat_25,feat_26,feat_27,feat_28,feat_29,feat_30,feat_31,PD_Label_Combined
0,0.265117,-0.010316,0.115100,0.328323,0.248238,-0.356049,-0.257898,0.174850,0.129193,0.233881,...,-0.247454,0.119079,-0.095544,-0.064415,0.132582,0.177669,0.097953,0.057453,-0.021450,1
1,0.263472,-0.014242,0.115215,0.324907,0.249789,-0.339662,-0.261713,0.168203,0.135401,0.235584,...,-0.245267,0.121652,-0.108255,-0.057349,0.149416,0.181106,0.106042,0.068035,-0.033311,1
2,0.235450,-0.175083,-0.047387,0.284162,0.299548,-0.321362,-0.242467,-0.029511,0.056156,0.208431,...,-0.348995,-0.075789,-0.172913,0.116194,-0.006837,0.078975,-0.079021,-0.094821,0.157325,0
3,0.265189,-0.012594,0.123403,0.323801,0.251000,-0.348127,-0.249223,0.160804,0.132252,0.234291,...,-0.251133,0.107482,-0.113234,-0.056090,0.151298,0.176699,0.085534,0.063102,-0.027583,1
4,0.265249,-0.010860,0.124201,0.326514,0.248276,-0.352222,-0.249442,0.164312,0.131111,0.233590,...,-0.253278,0.106324,-0.109191,-0.057897,0.144714,0.176427,0.082606,0.060780,-0.023348,1


In [14]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int[
    [f'feat_{i}' for i in range(32)]
].copy()

y = fused_df_named_int[
    "PD_Label_Combined"
].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00

Class Weights:
{0: 1.4621212121212122, 1: 0.7598425196850394}

Train Shape : (154, 32)
Test Shape  : (39, 32)

Training Class Distribution:
PD_Label_Combined
1    101
0     53
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    26
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(1.4528301886792452), np.int64(1): np.float64(0.7623762376237624)}

XGBoost scale_pos_weight: 0.5247524752475248


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 0.8125
Precision           : 1.0000
Sensitivity         : 0.7000
F1 Score            : 0.8235
ROC-AUC             : 0.9500
Balanced Accuracy   : 0.8500

XGBoost | Fold 2/10
Accuracy            : 0.9375
Precision           : 1.0000
Sensitivity         : 0.9000
F1 Score            : 0.9474
ROC-AUC         

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
HistGradientBoosting,0.824583,0.882449,0.851818,0.861917,0.909727,0.809242,0.846154,0.702705,0.927525,0.884615,...,0.497436,0.918205,0.884615,0.905325,0.826923,0.653846,10.0,3.0,3.0,23.0
Stacking Fusion,0.870000,0.938409,0.860909,0.895505,0.955000,0.872121,0.846154,0.702705,0.927525,1.000000,...,0.771905,1.000000,0.869565,0.928994,0.884615,0.725476,13.0,0.0,6.0,20.0
Random Forest,0.804583,0.840029,0.870000,0.851138,0.913576,0.771667,0.820513,0.673320,0.910227,0.880000,...,0.497436,0.918205,0.862745,0.914201,0.807692,0.604743,10.0,3.0,4.0,22.0
XGBoost,0.830833,0.899116,0.841818,0.865745,0.917545,0.822576,0.794872,0.644660,0.892203,0.875000,...,0.497436,0.918205,0.840000,0.926036,0.788462,0.559017,10.0,3.0,5.0,21.0
Voting Fusion,0.818333,0.869949,0.851818,0.857342,0.917061,0.800909,0.794872,0.644660,0.892203,0.875000,...,0.497436,0.918205,0.840000,0.914201,0.788462,0.559017,10.0,3.0,5.0,21.0


Without MIRA-Net

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_acoustic_top7 = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset_top7.csv')
print(df_acoustic_top7.shape)
print(df_acoustic_top7.head())

(193, 9)
       RPDE  Degree-voice breaks  Delta2  Delta3  Delta11  MFCC5   spread1  \
0  0.428488                  0.0     0.0     0.0      0.0    0.0 -4.239926   
1  0.542488                  0.0     0.0     0.0      0.0    0.0 -4.519291   
2  0.332247                  0.0     0.0     0.0      0.0    0.0 -7.226125   
3  0.498593                  0.0     0.0     0.0      0.0    0.0 -5.632290   
4  0.470775                  0.0     0.0     0.0      0.0    0.0 -5.523149   

  Participant_ID  Status  
0           d1_1       1  
1          d1_10       1  
2          d1_11       0  
3          d1_12       1  
4          d1_13       1  


In [3]:
fused_df_named_int = df_acoustic_top7.rename(columns={"Status": "PD_Label_Combined"})
fused_df_named_int = fused_df_named_int.drop(
    columns=["Participant_ID"]
).copy()
fused_df_named_int.head()
fused_df_named_int.shape

(193, 8)

In [4]:
fused_df_named_int.head()

,RPDE,Degree-voice breaks,Delta2,Delta3,Delta11,MFCC5,spread1,PD_Label_Combined
0,0.428488,0.0,0.0,0.0,0.0,0.0,-4.239926,1
1,0.542488,0.0,0.0,0.0,0.0,0.0,-4.519291,1
2,0.332247,0.0,0.0,0.0,0.0,0.0,-7.226125,0
3,0.498593,0.0,0.0,0.0,0.0,0.0,-5.632290,1
4,0.470775,0.0,0.0,0.0,0.0,0.0,-5.523149,1


In [5]:
# ============================================================
# ###8)###
# ENSEMBLE CLASSIFICATION WITH STRATIFIED 10-FOLD CV
# ============================================================

# ------------------ Install Required Packages ------------------

!pip install xgboost lightgbm catboost scikit-learn statsmodels matplotlib seaborn --quiet


# ------------------ Imports ------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from sklearn.ensemble import (
    VotingClassifier,
    StackingClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

from statsmodels.stats.proportion import proportion_confint


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

X = fused_df_named_int.iloc[:, :-1].copy()

y = fused_df_named_int["PD_Label_Combined"].copy()


# ------------------ Convert X to numeric ------------------

X = X.apply(
    pd.to_numeric,
    errors='coerce'
)

X.fillna(
    0,
    inplace=True
)


# ============================================================
# STEP 2: CLASS WEIGHTS
# ============================================================

classes = np.unique(y)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y
)

class_weights_dict = {
    int(classes[0]): float(class_weights[0]),
    int(classes[1]): float(class_weights[1])
}

print("\nClass Weights:")
print(class_weights_dict)


# ============================================================
# STEP 3: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


print("\nTrain Shape :", X_train.shape)
print("Test Shape  :", X_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


# ============================================================
# STEP 4: HANDLE CLASS IMBALANCE
# ============================================================

cw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

cw_dict = dict(
    zip(
        np.unique(y_train),
        cw
    )
)

scale_pos_weight = (
    cw[1] / cw[0]
)

print("\nTraining Class Weights:")
print(cw_dict)

print(
    "\nXGBoost scale_pos_weight:",
    scale_pos_weight
)


# ============================================================
# STEP 5: BASE MODEL CREATION FUNCTION
# ============================================================

def create_models():

    # ---------------- XGBoost ----------------

    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        eval_metric='logloss',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- Random Forest ----------------

    rf = RandomForestClassifier(
        n_estimators=200,

        class_weight='balanced',

        random_state=42,
        n_jobs=-1
    )


    # ---------------- HistGradientBoosting ----------------

    hgb = HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,

        class_weight='balanced',

        random_state=42
    )


    return xgb, rf, hgb


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CV
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


# ============================================================
# STEP 7: EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name):

    # ========================================================
    # CV METRIC STORAGE
    # ========================================================

    cv_accuracy = []
    cv_precision = []
    cv_sensitivity = []
    cv_f1 = []
    cv_auc = []
    cv_balanced_accuracy = []


    # ========================================================
    # 10-FOLD STRATIFIED CROSS-VALIDATION
    # ========================================================

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        print(
            f"\n{model_name} | Fold {fold}/10"
        )


        # ----------------------------------------------------
        # Split fold
        # ----------------------------------------------------

        X_tr = X_train.iloc[
            train_idx
        ]

        X_val = X_train.iloc[
            val_idx
        ]

        y_tr = y_train.iloc[
            train_idx
        ]

        y_val = y_train.iloc[
            val_idx
        ]


        # ----------------------------------------------------
        # Scaling INSIDE EACH FOLD
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_tr_scaled = scaler.fit_transform(
            X_tr
        )

        X_val_scaled = scaler.transform(
            X_val
        )


        # ----------------------------------------------------
        # Create FRESH models for every fold
        # ----------------------------------------------------

        xgb, rf, hgb = create_models()


        # ----------------------------------------------------
        # Select requested model
        # ----------------------------------------------------

        if model_name == "XGBoost":

            model = xgb


        elif model_name == "Random Forest":

            model = rf


        elif model_name == "HistGradientBoosting":

            model = hgb


        elif model_name == "Voting Fusion":

            model = VotingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],
                voting='soft'
            )


        elif model_name == "Stacking Fusion":

            model = StackingClassifier(
                estimators=[
                    ('xgb', xgb),
                    ('rf', rf),
                    ('hgb', hgb)
                ],

                final_estimator=LogisticRegression(
                    max_iter=1000,
                    class_weight='balanced'
                ),

                passthrough=True,

                cv=5
            )


        else:

            raise ValueError(
                f"Unknown model: {model_name}"
            )


        # ----------------------------------------------------
        # Train fold
        # ----------------------------------------------------

        model.fit(
            X_tr_scaled,
            y_tr
        )


        # ----------------------------------------------------
        # Validation predictions
        # ----------------------------------------------------

        val_preds = model.predict(
            X_val_scaled
        )

        val_probs = model.predict_proba(
            X_val_scaled
        )[:, 1]


        # ----------------------------------------------------
        # Fold metrics
        # ----------------------------------------------------

        fold_accuracy = accuracy_score(
            y_val,
            val_preds
        )

        fold_precision = precision_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_sensitivity = recall_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_val,
            val_preds,
            zero_division=0
        )

        fold_auc = roc_auc_score(
            y_val,
            val_probs
        )

        fold_balanced_accuracy = balanced_accuracy_score(
            y_val,
            val_preds
        )


        # ----------------------------------------------------
        # Store fold metrics
        # ----------------------------------------------------

        cv_accuracy.append(
            fold_accuracy
        )

        cv_precision.append(
            fold_precision
        )

        cv_sensitivity.append(
            fold_sensitivity
        )

        cv_f1.append(
            fold_f1
        )

        cv_auc.append(
            fold_auc
        )

        cv_balanced_accuracy.append(
            fold_balanced_accuracy
        )


        # ----------------------------------------------------
        # Print fold result
        # ----------------------------------------------------

        print(
            f"Accuracy            : {fold_accuracy:.4f}"
        )

        print(
            f"Precision           : {fold_precision:.4f}"
        )

        print(
            f"Sensitivity         : {fold_sensitivity:.4f}"
        )

        print(
            f"F1 Score            : {fold_f1:.4f}"
        )

        print(
            f"ROC-AUC             : {fold_auc:.4f}"
        )

        print(
            f"Balanced Accuracy   : "
            f"{fold_balanced_accuracy:.4f}"
        )


    # ========================================================
    # CV MEAN RESULTS
    # ========================================================

    cv_accuracy_mean = np.mean(
        cv_accuracy
    )

    cv_precision_mean = np.mean(
        cv_precision
    )

    cv_sensitivity_mean = np.mean(
        cv_sensitivity
    )

    cv_f1_mean = np.mean(
        cv_f1
    )

    cv_auc_mean = np.mean(
        cv_auc
    )

    cv_balanced_accuracy_mean = np.mean(
        cv_balanced_accuracy
    )


    # ========================================================
    # CROSS-VALIDATION RESULTS
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Cross-Validation (10-fold)"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : "
        f"{cv_accuracy_mean:.4f}"
    )

    print(
        f"Precision           : "
        f"{cv_precision_mean:.4f}"
    )

    print(
        f"Recall/Sensitivity  : "
        f"{cv_sensitivity_mean:.4f}"
    )

    print(
        f"F1 Score            : "
        f"{cv_f1_mean:.4f}"
    )

    print(
        f"ROC-AUC             : "
        f"{cv_auc_mean:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{cv_balanced_accuracy_mean:.4f}"
    )


    # ========================================================
    # TRAIN FINAL MODEL ON COMPLETE TRAINING SET
    # ========================================================

    final_scaler = StandardScaler()

    X_train_final_scaled = final_scaler.fit_transform(
        X_train
    )

    X_test_final_scaled = final_scaler.transform(
        X_test
    )


    # --------------------------------------------------------
    # Create fresh final models
    # --------------------------------------------------------

    xgb, rf, hgb = create_models()


    if model_name == "XGBoost":

        final_model = xgb


    elif model_name == "Random Forest":

        final_model = rf


    elif model_name == "HistGradientBoosting":

        final_model = hgb


    elif model_name == "Voting Fusion":

        final_model = VotingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],
            voting='soft'
        )


    elif model_name == "Stacking Fusion":

        final_model = StackingClassifier(
            estimators=[
                ('xgb', xgb),
                ('rf', rf),
                ('hgb', hgb)
            ],

            final_estimator=LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            ),

            passthrough=True,

            cv=5
        )


    # ========================================================
    # FINAL TRAINING
    # ========================================================

    final_model.fit(
        X_train_final_scaled,
        y_train
    )


    # ========================================================
    # TEST PREDICTIONS
    # ========================================================

    test_preds = final_model.predict(
        X_test_final_scaled
    )

    test_probs = final_model.predict_proba(
        X_test_final_scaled
    )[:, 1]


    # ========================================================
    # TEST METRICS
    # ========================================================

    accuracy = accuracy_score(
        y_test,
        test_preds
    )

    precision = precision_score(
        y_test,
        test_preds,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_preds,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_preds,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probs
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_preds
    )

    mcc = matthews_corrcoef(
        y_test,
        test_preds
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    cm = confusion_matrix(
        y_test,
        test_preds,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()


    # ========================================================
    # SENSITIVITY / SPECIFICITY
    # ========================================================

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )


    # ========================================================
    # 95% CONFIDENCE INTERVALS
    # ========================================================

    # ---------------- Accuracy CI ----------------

    acc_low, acc_high = proportion_confint(
        count=int(round(accuracy * len(y_test))),
        nobs=len(y_test),
        alpha=0.05,
        method='wilson'
    )


    # ---------------- Precision CI ----------------

    precision_low, precision_high = proportion_confint(
        count=tp,
        nobs=(tp + fp),
        alpha=0.05,
        method='wilson'
    ) if (tp + fp) > 0 else (0.0, 0.0)


    # ---------------- Sensitivity CI ----------------

    sensitivity_low, sensitivity_high = proportion_confint(
        count=tp,
        nobs=(tp + fn),
        alpha=0.05,
        method='wilson'
    ) if (tp + fn) > 0 else (0.0, 0.0)


    # ---------------- Specificity CI ----------------

    specificity_low, specificity_high = proportion_confint(
        count=tn,
        nobs=(tn + fp),
        alpha=0.05,
        method='wilson'
    ) if (tn + fp) > 0 else (0.0, 0.0)


    # ========================================================
    # TEST SET PERFORMANCE
    # ========================================================

    print(
        "\n================================================"
    )

    print(
        f"{model_name} - Test Set Performance"
    )

    print(
        "================================================"
    )

    print(
        f"Accuracy            : {accuracy:.4f} "
        f"(95% CI: "
        f"{acc_low:.4f}-{acc_high:.4f})"
    )

    print(
        f"Precision           : {precision:.4f} "
        f"(95% CI: "
        f"{precision_low:.4f}-{precision_high:.4f})"
    )

    print(
        f"Sensitivity         : {sensitivity:.4f} "
        f"(95% CI: "
        f"{sensitivity_low:.4f}-{sensitivity_high:.4f})"
    )

    print(
        f"Specificity         : {specificity:.4f} "
        f"(95% CI: "
        f"{specificity_low:.4f}-{specificity_high:.4f})"
    )

    print(
        f"F1 Score            : {f1:.4f}"
    )

    print(
        f"ROC-AUC             : {roc_auc:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"MCC                 : {mcc:.4f}"
    )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    print(
        "\nConfusion Matrix:"
    )

    print(cm)

    print(
        "\nTN :", tn
    )

    print(
        "FP :", fp
    )

    print(
        "FN :", fn
    )

    print(
        "TP :", tp
    )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    result = {

        # ---------------- CV ----------------

        "CV Accuracy":
            cv_accuracy_mean,

        "CV Precision":
            cv_precision_mean,

        "CV Sensitivity":
            cv_sensitivity_mean,

        "CV F1":
            cv_f1_mean,

        "CV ROC-AUC":
            cv_auc_mean,

        "CV Balanced Accuracy":
            cv_balanced_accuracy_mean,


        # ---------------- Test ----------------

        "Accuracy":
            accuracy,

        "Accuracy CI Low":
            acc_low,

        "Accuracy CI High":
            acc_high,


        "Precision":
            precision,

        "Precision CI Low":
            precision_low,

        "Precision CI High":
            precision_high,


        "Sensitivity":
            sensitivity,

        "Sensitivity CI Low":
            sensitivity_low,

        "Sensitivity CI High":
            sensitivity_high,


        "Specificity":
            specificity,

        "Specificity CI Low":
            specificity_low,

        "Specificity CI High":
            specificity_high,


        "F1":
            f1,

        "ROC-AUC":
            roc_auc,

        "Balanced Accuracy":
            balanced_accuracy,

        "MCC":
            mcc,


        # ---------------- Confusion Matrix ----------------

        "TN":
            tn,

        "FP":
            fp,

        "FN":
            fn,

        "TP":
            tp
    }


    return result


# ============================================================
# STEP 8: EVALUATE ALL MODELS
# ============================================================

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]


results = {}

for name in model_names:

    print(
        "\n\n################################################"
    )

    print(
        f"Evaluating: {name}"
    )

    print(
        "################################################"
    )

    results[name] = evaluate_model(
        name
    )


# ============================================================
# STEP 9: FINAL RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
).T


# ============================================================
# STEP 10: ORDER RESULTS
# ============================================================

results_df = results_df[
    [

        # ---------------- CV ----------------

        "CV Accuracy",
        "CV Precision",
        "CV Sensitivity",
        "CV F1",
        "CV ROC-AUC",
        "CV Balanced Accuracy",


        # ---------------- Test ----------------

        "Accuracy",
        "Accuracy CI Low",
        "Accuracy CI High",

        "Precision",
        "Precision CI Low",
        "Precision CI High",

        "Sensitivity",
        "Sensitivity CI Low",
        "Sensitivity CI High",

        "Specificity",
        "Specificity CI Low",
        "Specificity CI High",

        "F1",
        "ROC-AUC",
        "Balanced Accuracy",
        "MCC",

        # ---------------- Confusion Matrix ----------------

        "TN",
        "FP",
        "FN",
        "TP"
    ]
]


# ============================================================
# STEP 11: DISPLAY FINAL RESULTS
# ============================================================

print(
    "\n================================================"
)

print(
    "FINAL MODEL COMPARISON"
)

print(
    "================================================\n"
)


display(
    results_df.sort_values(
        by="F1",
        ascending=False
    )
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.8 MB/s eta 0:00:00

Class Weights:
{0: 1.4621212121212122, 1: 0.7598425196850394}

Train Shape : (154, 7)
Test Shape  : (39, 7)

Training Class Distribution:
PD_Label_Combined
1    101
0     53
Name: count, dtype: int64

Testing Class Distribution:
PD_Label_Combined
1    26
0    13
Name: count, dtype: int64

Training Class Weights:
{np.int64(0): np.float64(1.4528301886792452), np.int64(1): np.float64(0.7623762376237624)}

XGBoost scale_pos_weight: 0.5247524752475248


################################################
Evaluating: XGBoost
################################################

XGBoost | Fold 1/10
Accuracy            : 0.9375
Precision           : 1.0000
Sensitivity         : 0.9000
F1 Score            : 0.9474
ROC-AUC             : 1.0000
Balanced Accuracy   : 0.9500

XGBoost | Fold 2/10
Accuracy            : 0.9375
Precision           : 1.0000
Sensitivity         : 0.9000
F1 Score            : 0.9474
ROC-AUC           

,CV Accuracy,CV Precision,CV Sensitivity,CV F1,CV ROC-AUC,CV Balanced Accuracy,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,...,Specificity CI Low,Specificity CI High,F1,ROC-AUC,Balanced Accuracy,MCC,TN,FP,FN,TP
XGBoost,0.830833,0.910260,0.831818,0.864771,0.931045,0.827576,0.846154,0.702705,0.927525,0.884615,...,0.497436,0.918205,0.884615,0.905325,0.826923,0.653846,10.0,3.0,3.0,23.0
Random Forest,0.778750,0.833947,0.830909,0.828904,0.894682,0.752121,0.846154,0.702705,0.927525,0.884615,...,0.497436,0.918205,0.884615,0.894970,0.826923,0.653846,10.0,3.0,3.0,23.0
Voting Fusion,0.817917,0.894545,0.830909,0.854185,0.914212,0.807121,0.846154,0.702705,0.927525,0.884615,...,0.497436,0.918205,0.884615,0.926036,0.826923,0.653846,10.0,3.0,3.0,23.0
HistGradientBoosting,0.799583,0.913409,0.781818,0.826317,0.903045,0.802576,0.846154,0.702705,0.927525,0.954545,...,0.666860,0.986290,0.875000,0.893491,0.865385,0.694709,12.0,1.0,5.0,21.0
Stacking Fusion,0.831250,0.932045,0.812727,0.863497,0.928848,0.838030,0.769231,0.616637,0.873542,0.869565,...,0.497436,0.918205,0.816327,0.905325,0.769231,0.516047,10.0,3.0,6.0,20.0
